# 🐖 08 | Comparación exploratoria YOLO versus Mask R-CNN

**RECURSO: CPU — SIN ACELERADOR**. Lee métricas ya guardadas; no entrena ni hace inferencia sobre imágenes. Exporta Excel de comparación, figura y conclusiones metodológicas.

**Proyecto:** Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖

Los notebooks son independientes: cada uno utiliza los productos guardados por la etapa anterior, sin repetir descarga o entrenamiento.


## Objetivo

Comparar de manera descriptiva **YOLO11n-seg** y **Mask R-CNN ResNet50-FPN** en el conjunto de 373 imágenes test reservadas por fecha. Los resultados son **provisionales**: las métricas Ultralytics y COCOeval tienen diferencias técnicas; las épocas y los presupuestos de cómputo también difieren. No extraer conclusiones de superioridad ni significación estadística hasta realizar evaluación armonizada y varias semillas.

Si todavía no ejecutaste Mask R-CNN (notebook 06 y 07), este notebook te avisará sin crear resultados ficticios.


In [ ]:
%pip -q install pandas openpyxl matplotlib pyyaml


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, os

drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Proyecto de investigación: Visión Artificial y Deep Learning en Cerdos 🐖')
if not ROOT.is_dir():
    raise FileNotFoundError('No encuentro la carpeta del proyecto. Comprueba que Colab monta la cuenta correcta y que el nombre no cambió: '+str(ROOT))
DATA=ROOT/'01_Datasets'/'FaroPigSeg'
EXTRACT=DATA/'extraido'/'FaroPigSeg'
SPLIT=DATA/'split_fecha_v1'
RES=ROOT/'03_Resultados'
EXP=ROOT/'05_Exportaciones_ZIP'
for s in ['01_Exploracion','02_Segmentacion','03_Metricas','04_Figuras','05_Reportes','06_Entrenamiento']:
    (RES/s).mkdir(parents=True,exist_ok=True)
EXP.mkdir(exist_ok=True,parents=True)
MANIFEST=SPLIT/'particion_fecha_manifest.json'
YAML=SPLIT/'data_fecha.yaml'
if not (MANIFEST.is_file() and YAML.is_file()):
    raise FileNotFoundError('Faltan los archivos de partición congelada; ejecutar notebook 02 en CPU antes de continuar.')
part=json.loads(MANIFEST.read_text(encoding='utf-8'))
assert len(set(part['asignaciones_fecha'].values())-{'train','val','test'})==0
print('Carpeta del proyecto:',ROOT)
print('Fechas por grupo:',{k:sum(v==k for v in part['asignaciones_fecha'].values()) for k in ('train','val','test')})


In [ ]:
import pandas as pd,numpy as np,matplotlib.pyplot as plt
YOLO_REPORT=RES/'03_Metricas'/'CERDOS_04_Evaluacion_test.json'
MASK_REPORT=RES/'05_Reportes'/'CERDOS_07_Evaluacion_MaskRCNN.json'
MASK_EXCEL=RES/'03_Metricas'/'CERDOS_07_Por_Imagen_MaskRCNN.xlsx'
for p in [YOLO_REPORT,MASK_REPORT,MASK_EXCEL]:
    if not p.is_file():raise FileNotFoundError(f'Falta {p.name}; primero finaliza 04 (YOLO) y 07 (Mask R-CNN).')
yolo=json.loads(YOLO_REPORT.read_text(encoding='utf8'))
mask=json.loads(MASK_REPORT.read_text(encoding='utf8'))
assert int(yolo['test_count'])==int(mask['num_images']), 'Los test no parecen iguales; detener comparación.'
assert mask['yaml_sha256']==hashlib.sha256(YAML.read_bytes()).hexdigest(), 'Cambiaron las listas de test; detener.'
metrics=['seg_map50','seg_map50_95','seg_map75']
rows=[]
for model,rec,source in [('YOLO11n-seg',yolo,'Ultralytics'),('Mask R-CNN ResNet50-FPN',mask,'COCOeval')]:
    rows.append({'arquitectura':model,'evaluador':source,'test_imagenes':int(rec.get('test_count',rec.get('num_images'))),
                 'mAP50_mascaras':float(rec['seg_map50']), 'mAP50_95_mascaras':float(rec['seg_map50_95']),
                 'mAP75_mascaras':float(rec.get('seg_map75',np.nan)),
                 'peso_sha256':rec['weights_sha256']})
compare=pd.DataFrame(rows)
delta=float(compare.iloc[1]['mAP50_95_mascaras']-compare.iloc[0]['mAP50_95_mascaras'])
print(compare[['arquitectura','evaluador','test_imagenes','mAP50_mascaras','mAP50_95_mascaras']].to_string(index=False))
print('Diferencia exploratoria Mask R-CNN − YOLO (mAP50-95) =',round(delta,4))
print('⚠️ Evaluadores diferentes. No afirmar ventaja estadística; el test no es para sintonizar hiperparámetros.')


In [ ]:
report=RES/'05_Reportes'/'CERDOS_08_Conclusiones_Preliminares.json'
excel=RES/'03_Metricas'/'CERDOS_08_Comparacion_Arquitecturas.xlsx'
err=pd.read_excel(MASK_EXCEL,sheet_name='Errores_por_fecha')
notes=[
 'FaroPigSeg tiene imágenes; esta comparación no demuestra seguimiento ni conducta.',
 'Las 12 figuras del ZIP YOLO eran ejemplos; el test cuantitativo abarcó 373 imágenes.',
 'Se estiman AP por evaluadores distintos (Ultralytics y COCOeval); comparación provisional.',
 'Los modelos tienen diferente número de parámetros, épocas y presupuesto de cómputo.',
 'Sin intervalos de incertidumbre armonizados y varias semillas no afirmar superioridad.',
 'La independencia entre fechas no implica independencia entre granjas o animales.',
 'Medición de instancias por imagen no es identificación única de cerdos.'
]
with pd.ExcelWriter(excel,engine='openpyxl') as w:
    compare.to_excel(w,sheet_name='Comparacion_global',index=False)
    err.to_excel(w,sheet_name='Errores_MaskRCNN_fecha',index=False)
    pd.DataFrame({'limitaciones':notes}).to_excel(w,sheet_name='Limitaciones',index=False)
fig,ax=plt.subplots(figsize=(9,4.6))
x=np.arange(len(compare));width=.32
ax.bar(x-width/2,compare['mAP50_mascaras'],width,label='mAP50')
ax.bar(x+width/2,compare['mAP50_95_mascaras'],width,label='mAP50-95')
ax.set_xticks(x,compare['arquitectura']);ax.set_ylabel('AP (0–1)');ax.set_ylim(0,1)
ax.set_title('Comparación descriptiva en test reservado por fecha')
ax.legend();fig.tight_layout()
figure=RES/'04_Figuras'/'CERDOS_08_AP_Arquitecturas.png';fig.savefig(figure,dpi=180);plt.show()
report.write_text(json.dumps({'fecha_utc':datetime.now(timezone.utc).isoformat(),
             'evaluacion':'preliminar_no_armonizada','metricas':compare.to_dict('records'),
             'diferencia_maskrcnn_menos_yolo':delta,'limitaciones':notes},
             ensure_ascii=False,indent=2),encoding='utf8')
print('✅ Excel:',excel)


In [ ]:
import zipfile
from google.colab import files as colab_files
export=EXP/'CERDOS_08_Comparacion_Preliminar.zip'
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in [excel,figure,report]:z.write(file,arcname=file.name)
print('✅ ZIP:',export)
colab_files.download(str(export))


### Próxima fase científica

Para resultados publicables de comparación, una siguiente etapa deberá armonizar métricas por el mismo evaluador, analizar incertidumbre por fechas, múltiples semillas y perfiles de cómputo. **Videos de cerdos con ID temporal** serán necesarios para seguimiento y extracción de comportamiento.
